# Admixture proportions from low depth sequencing (NGSadmix)

**Purpose.** Estimate how much of each individual's genome comes from each of K ancestral
populations, working from **genotype likelihoods** so that no genotype ever has to be
called. At a few reads per site a called genotype is often wrong, and the error depends
on depth — which would show up as fake structure.

**What you will do**
 - run NGSadmix on a small toy dataset and read its output
 - run it on real 1000 Genomes data and plot the admixture proportions
 - assess model fit with evalAdmix, and see what a bad fit looks like
 - try several values of K and several seeds, and decide which run to keep

**The data.** 435 individuals from the **1000 Genomes Project**, five populations:

| Code | Population | n |
|---|---|---|
| ASW | African ancestry in the southwestern USA | 61 |
| CEU | Utah residents with northern/western European ancestry | 99 |
| CHB | Han Chinese in Beijing, China | 103 |
| MXL | Mexican ancestry in Los Angeles, USA | 64 |
| YRI | Yoruba in Ibadan, Nigeria | 108 |

Supplied as **genotype likelihoods** in beagle format, not called genotypes. ASW and MXL
are the admixed populations — they are the reason the exercise exists.

**Before this** it helps to have done [the EM algorithm behind ADMIXTURE](../em_algorithms/admixture_em_human.ipynb).

## Setup

All the paths used by this exercise are set in the cell below.

In [ ]:
#############################################################
# ALL PATHS ARE SET HERE
# If the data or the software moves, this is the ONLY cell you
# need to change. No cell below this one uses a full path.
#############################################################

# where the shared data lives
DATA=/course/data/current_data/admixture/human_lowdepth

# where you will do the exercise
WORK_DIR=$HOME/admixture_low_depth_human

mkdir -p $WORK_DIR
echo $WORK_DIR > $HOME/.admixture_lowdepth_workdir
cd $WORK_DIR

# link the input files into the working folder
cp -sf $DATA/1000G5pops.inputgl.beagle.gz .
cp -sf $DATA/1000G5pops.pop.info .
cp -r -sf $DATA/admixoutput .
cp -r -sf $DATA/evaladmixoutput .

echo --programs that are installed:--
which NGSadmix
which evalAdmix

echo; echo --- files in folder ---
ls

In [ ]:
# the working directory was set in the first cell of the notebook
work_d <- readLines(path.expand("~/.admixture_lowdepth_workdir"))[1]
setwd(work_d)
getwd()

## First small example

We will first try to run an NGSadmix analysis of a small dataset consisting of bam files with low depth NGS data from 435 samples of 5 human populations from the 1000 genomes project:


| Population code | Population                                     | Sample size |
|-----------------|------------------------------------------------|-------------|
| ASW             | HapMap African ancestry individuals from SW US | 61          |
| CEU             | European individuals                           | 99          |
| CHB             | Han Chinese in Beijing                         | 103         |
| YRI             | Yoruba individuals from Nigeria                | 108         |
| MXL             | Mexican individuals from LA California         | 64          |


### Make input data using ANGSD

The input to NGSadmix is genotype likelihoods (GLs). Therefore the first step of running an NGSadmix analysis (if all you have are bams files) is to calculate GLs. So let's start bying doing that. First make a file that contains the paths of all the bam files by finding all the names of the bamfiles in the relevant folder and saving this in a file called all.files:

### Why genotype likelihoods and not genotypes

These samples are sequenced to a few reads per site. At that depth a large fraction of the sites in
an individual are covered by a single read, and a single read can never distinguish a heterozygote
from a homozygote - so calling the genotype would systematically turn heterozygotes into
homozygotes. Instead of calling, ANGSD computes for every site and every individual the likelihood
of each of the three possible genotypes given the reads and their base qualities. A site with one
read gives a flat, uninformative likelihood; a site with ten reads gives a sharp one. NGSadmix then
works directly on these likelihoods, so the uncertainty is part of the model rather than a source of
bias.

In [ ]:
source env.sh   # paths + cd to the working folder

find $BAMFOLDER | grep bam$ > all.files


To see the content of the file you made type:

In [ ]:
source env.sh   # paths + cd to the working folder

cat all.files

As you can see it is a list of paths to bams files (a line per bam file). Each bam file contains data from a single individual and the population a given individual is from is indicated as a part of the bam file name (if we divide the path/file name into text bites between "." it is the 6th text bite). To see how many individuals there is data for from each population we can therefore count how many bam file names there are with each of the different populations as part of their name:

In [ ]:
source env.sh   # paths + cd to the working folder

cat all.files | cut -d'.' -f6 | sort | uniq -c

- Does this fit the info you were given above about the dataset? (Which populations, number of individuals from each of the populations, total number of individuals)

Now calculate GLs from all the BAM files using ANGSD by running the following command in the terminal:

Here is what the options mean:

```
-bam all.files   : the bam files to calculate genotype likelihoods from are listed in all.files
-GL 2            : use the GATK genotype likelihood model
-doMajorMinor 1  : infer the major and minor allele from the data
-doMaf 1         : estimate the minor allele frequency (needed by -SNP_pval and -minMaf below)
-SNP_pval 2e-6   : only keep sites that are variable with a p-value below 2e-6
-minMapQ 30      : discard reads with a mapping quality below 30
-minQ 20         : discard bases with a base quality below 20
-minInd 25       : only keep sites with data for at least 25 individuals
-minMaf 0.05     : only keep sites with a minor allele frequency above 0.05
-doGlf 2         : write the genotype likelihoods in beagle format
-out all         : name all output files "all" followed by the appropriate suffix
-P 5             : use 5 threads
```

Note that we are calling SNPs and filtering on allele frequency *across all 435 samples at once*.
That is deliberate: the sites we keep should not depend on which population an individual came from.

In [ ]:
source env.sh   # paths + cd to the working folder

$ANGSD -bam all.files -GL 2 -doMajorMinor 1 -doMaf 1 -SNP_pval 2e-6 -minMapQ 30 -minQ 20 -minInd 25 -minMaf 0.05 -doGlf 2 -out all -P 5


NOTE that this will take a bit of time to run (around a minute). While waiting, try to remember what the different options used mean (you have seen most of them in previous exercises). If you do not remember all of them, then try to ask the person next to you. And if neither of you remember then try to figure it out by looking for help on the ANGSD webpage e.g. [here](http://www.popgen.dk/angsd/index.php/Genotype_Likelihoods), [here](http://www.popgen.dk/angsd/index.php/Major_Minor) and [here](http://www.popgen.dk/angsd/index.php/Filters).



## Explore the input data

Now let's have a look at the GL file that you have created with ANGSD. It is a "beagle format" file called all.beagle.gz - and will be the input file to NGSadmix. The first line in this file is a header line and after that it contains a line for each locus with GLs. By using the unix command wc we can count the number of lines in the file:

In [ ]:
source env.sh   # paths + cd to the working folder

gunzip -c all.beagle.gz | wc -l


- Use this to find out how many loci there are GLs for in the data set?

Next, to get an idea of what the GL file contains try from the command line to print the first 9 columns of the first 7 lines of the file:

In [ ]:
source env.sh   # paths + cd to the working folder

zcat all.beagle.gz | awk '(NR<=7)' | cut -f1-9 | column -t


**Questions**
 - A beagle file has three numbers per individual per site. What do they add up to, and what do they mean?
 - How many sites are in this file?

In general, the first three columns of a beagle file contain the locus name, and the two alleles, allele1 and allele2, present in that locus (in beagle A=0, C=1, G=2, T=3). All following columns contain genotype likelihoods (three columns for each individual: first GL for homozygote for allele1, then GL for heterozygote and then GL for homozygote for allele2). Note that the GL values sum to one per site for each individuals. This is just a normalization of the genotype likelihoods in order to avoid underflow problems in the beagle software it does not mean that they are genotype probabilities. 

Based on this: 

- What is the name of the first locus?
- Which nucleotide is allele2 in the first locus?
- What is the most likely genotype for ind0 in the first locus? 

### Quiz

In [ ]:
from jupyterquiz import display_quiz

display_quiz("https://raw.githubusercontent.com/popgenDK/courses/main/current_exercises/admixture/quiz/admix_quiz1.json")


## Run an analysis of the data with NGSadmix

Now you know how the input looks. Next, let's try to perform an NGSadmix analyses of the GLs typing assuming the number of ancestral populations, K, is 3:

The options are:

```
-likes all.beagle.gz : the genotype likelihoods to analyse
-K 3                 : assume 3 ancestral populations
-minMaf 0.05         : ignore sites where the estimated minor allele frequency is below 0.05
-seed 1              : the seed for the random starting point of the optimisation
-o all               : prefix for the output files
```

Sites with a very low minor allele frequency carry almost no information about ancestry but still
cost time, which is why they are filtered out. The `-seed` is the one to keep an eye on - it decides
where the optimisation starts, and we will come back to it.

In [ ]:
source env.sh   # paths + cd to the working folder

$NGSadmix -likes all.beagle.gz -K 3 -minMaf 0.05 -seed 1 -o all


**Questions**
 - `-K 3` asks for three ancestral populations. Where did that number come from?
 - `-minMaf 0.05` drops rare variants. Why would rare variants be unhelpful here?

- When it is done you will see some output above. While waiting for the analysis to finish running please make sure you understand the command you ran. If you are in doubt seek help [here](http://www.popgen.dk/software/index.php/NgsAdmix#Parameters). Here you can also see what other options you have when you run an NGSadmix analyses.


## Explore the output

The output from the analysis you just ran is three files:

- all.log (a "log file" that summarizes the analysis run)
- all.fopt.gz (an "fopt file", which has a line for each locus that contains an estimate of the allele frequency in each of the 3 assumed ancestral populations)
- all.qopt (a "qopt file", which has a line for each individual that contains anestimate of the individual's ancestry proportion from each of the three assumed ancestral populations).

Let's have a look at them one at a time. First, check the log file by typing

In [ ]:
source env.sh   # paths + cd to the working folder

cat all.log

- What is the log likelihood of the estimates achieved by NGSadmix (called "best like" in the log file)?

Next, check the first line of the fopt file by typing:

In [ ]:
source env.sh   # paths + cd to the working folder

zcat all.fopt.gz | awk 'NR==1'

- Based on this: what is the estimated allele frequency at the first locus in the three assumed ancestral populations?

Finally, check the 6th line of the qopt file and thus the estimated admixture proportions for the 6th individual by typing:

In [ ]:
source env.sh   # paths + cd to the working folder

head -n6 all.qopt | tail -n1

- Based on this: does the individual look admixed?

You can see the ID of the 6th individual by getting the 6th line of the file you created with all your original bam files in the beginning:

In [ ]:
source env.sh   # paths + cd to the working folder

head -n6 all.files | tail -n1 

- Based on that ID, which population does the individual come from?
- Based on this and the frequency estimates for the first locus that you looked at earlier, what does NGSadmix estimate the allele frequency to be at the first locus in that population?

## Plot the admixture proportion estimates

Finally, let us plot the estimated admixture proportions for all the individuals. The next cell runs in R.

In [ ]:

# Get ID and pop info for each individual
s<-strsplit(basename(scan("all.files",what="theFuck")),"\\.")
pop<-sapply(s,function(x) x[6])

# Import some functions to help in visualization
source("https://raw.githubusercontent.com/GenisGE/evalAdmix/master/visFuns.R")

# Read in inferred admixture proportions
q<-read.table("all.qopt")

# Order individuals by population, and within population by admixture proportion
ord <- orderInds(pop=pop,q=q, popord=c("YRI", "ASW", "CEU", "MXL", "CHB"))

# Make plot            
par(mar=c(7,4,1,1))
barplot(t(q)[,ord],col=c(3,2,4),las=2,xlab="Individuals",ylab="Admixture proportions", space=0, border=NA)
text(sort(tapply(1:length(pop),pop[ord],mean)),-0.05,unique(pop[ord]),xpd=NA)
abline(v=cumsum(sapply(unique(pop[ord]),function(x){sum(pop[ord]==x)})),col=1,lwd=1.2)
            

Note that the order of the individuals in the plot are not the same as in the qopt file. Instead, to provide a better overview, the individuals have been ordered according to the population they are sampled from.

- Try to explain what the plot shows (what is on the axes, what do the colors mean and so on)
- What does the plot suggest about whether the individuals are admixed?

NB As you could tell from the number of loci included in the analysis, the above analysis is based on data from very few loci (actually we on purpose only analyzed data from a small part of the genome to make sure the analysis ran fast). In the following we will redo the analyses using a larger number of sites.


### Quiz

In [ ]:
from jupyterquiz import display_quiz

display_quiz("https://raw.githubusercontent.com/popgenDK/courses/main/current_exercises/admixture/quiz/admix_quiz2.json")


## More realistic example

Now you know how to make input data to NGSadmix, how to run NGSadmix and what the output looks like. We will try now to run a more realistic dataset, using the same samples with a larger number of sites. We have already made the input file with genotype likelihoods for 100 000 sites across the genome, and a file with population info.


- the genotype likelihoods of all 435 individuals in beagle format: `data/admixinput/1000G5pops.inputgl.beagle.gz`
- a file with labels that indicate which population each of them is sampled from: `data/admixinput/1000G5pops.pop.info`


## Run an analysis of the data with NGSadmix

We start by running an NGSadmix analyses with K=3 (-K 3), using 10 cpu threads (-P 10) and using only SNPs with minor allele frequency above 0.05 (-minMaf 0.05). Furthermore, to make sure we reach the maximum likelihood solution and not a local optima, we should run 20 independent optimizations runs (-seed i for i in 1:20).

(NB. Because running this would be too computationally intense to run everyone at the same time on the server, we have already run it and the following code just prints the commands you would need to run).  

### Why one run is not enough

NGSadmix maximises the likelihood by starting from a random guess for `f` and `Q` and improving it
step by step. Each step increases the likelihood, so the algorithm always ends somewhere it cannot
improve on locally - but that place need not be the best solution overall. Where it ends depends on
where it started, and where it starts is decided by the seed.

The only practical defence is to run the same analysis from many different starting points and
compare the likelihoods they reach. If most runs reach the same, highest value, we have reasonable
confidence that this is the global maximum. If every run ends somewhere different, the estimate
should not be trusted - and note that you cannot average the runs, because the K clusters come out in
an arbitrary order and the first column of one run is not the same ancestral population as the first
column of another.

In [ ]:
source env.sh   # paths + cd to the working folder

inputpath=1000G5pops.inputgl.beagle.gz
outpath=admixoutput
K=3

for i in `seq 1 20`
do
    echo "$NGSadmix -likes $inputpath -K $K -P 10 -minMaf 0.05 -seed $i -o ${outpath}/1000G5popsAdmixK${K}seed${i}"
done


**Question**
 - This run uses the real 1000 Genomes data rather than the toy set. How long does it take, and what dominates that time — the number of individuals or the number of sites?

This will produce 20 NGSadmix results with their corresponding output files. In order to assess convergence and find the run with the best log likelihood, we need to check the log likelihoods of the data. This command with extract the log likelihood of each run from the log file, add the seed and sort them from the best to the worse log likelihood.

In [ ]:
source env.sh   # paths + cd to the working folder

outadmix=admixoutput/1000G5popsAdmix
K=3

rm -f allK$K.likes
for i in `seq 1 20`
do 
    cat ${outadmix}K${K}seed$i.log | grep "best like" | awk -F"[ =]" '{print $3}' >> allK$K.likes
done

cat -n allK$K.likes | sort -rhk2

### Quiz

In [ ]:
from jupyterquiz import display_quiz

display_quiz("https://raw.githubusercontent.com/popgenDK/courses/main/current_exercises/admixture/quiz/admix_quiz3.json")


- Does it look convergence was reached?

We continue by visualizing the results of the maximum likelihood run

In [ ]:

# Import some functions to help in visualization
source("https://raw.githubusercontent.com/GenisGE/evalAdmix/master/visFuns.R") 

# Read in info to plot
pop<-read.table("1000G5pops.pop.info",as.is=T)
q<-read.table("admixoutput/1000G5popsAdmixK3seed3.qopt")

# Sort individuals by population and within populations by admixture proportion
ord<-orderInds(pop = pop[,1], q=q) 

# Make plot
barplot(t(q)[,ord],col=2:10,space=0,border=NA,xlab="Individuals",ylab="Admixture proportions")
text(sort(tapply(1:nrow(pop),pop[ord,1],mean)),-0.05,unique(pop[ord,1]),xpd=T) # add population labels
abline(v=cumsum(sapply(unique(pop[ord,1]),function(x){sum(pop[ord,1]==x)})),col=1,lwd=1.2)


- Why do you think it looks cleaner than the previous admixture plot we visualized with the same individuals?
- How many populations would you now say are admixed? Which populations seem to be the admixture sources? Does that make sense given what you know of these populations?

## Assessing model fit

We will now use evalAdmix to assess if the ancestries inferred in our admixture results are a good approximation to the correct ancestries.

Again, due to time and computational limitations, we have already ran the command and just provide here the command used:

### How to read the correlation of residuals

The admixture model predicts, for each individual and site, an expected genotype - its own mixture of
the estimated ancestral frequencies. The **residual** is the difference between what we observed and
that expectation. If the model is right, the residuals are noise: sequencing error and the randomness
of which alleles an individual happened to inherit. Noise is independent between individuals, so the
correlation between the residuals of any two individuals should be about 0.

If it is not, the two individuals deviate from the model *in the same direction*, and they do so
because they share something the model does not know about. In the heatmap:

- a **block of positive correlation** covering a whole population means that population has ancestry
  that the K assumed ancestral populations cannot express, and it is being forced into a mixture of
  the ones we do have. Raising K is the usual remedy.
- a **single bright pair** off the diagonal usually means those two individuals are closely related.
- **negative correlation** between two groups is the mirror image of a positive block elsewhere - if
  some individuals are pulled one way, others must be pulled the other way for the frequencies to
  add up.

In [ ]:
source env.sh   # paths + cd to the working folder

K=3
besti=3
inbgl=1000G5pops.inputgl.beagle.gz
inadmix=admixoutput/1000G5popsAdmixK${K}seed${besti}
out=evaladmixoutput/1000G5pops.K${K}seed${besti}.corres

echo "$EVALADMIX -beagle $inbgl -fname ${inadmix}.fopt.gz -qname ${inadmix}.qopt -o $out -P 20"


**Questions**
 - evalAdmix plots the correlation of residuals between pairs of individuals. What should that look like if the model fits?
 - Which populations show the worst fit, and what does that suggest?

We will now visualize the correlation of residuals estimated by evalAdmix, and use it to assess whether the estimated admixture proportions results are a good fit to the data.

In [ ]:

# Import some funcitons to help in visualization
source("https://raw.githubusercontent.com/GenisGE/evalAdmix/master/visFuns.R")

# Read in info to plot
pop<-read.table("1000G5pops.pop.info",as.is=T)
q<-read.table("admixoutput/1000G5popsAdmixK3seed3.qopt")
r <- as.matrix(read.table("evaladmixoutput/1000G5pops.K3seed3.corres"))

# Sort individuals by population and within populations by admixture proportion
ord<-orderInds(pop = pop[,1], q=q) 

# Make plot
plotCorRes(r, pop=pop[,1], ord=ord, max_z = 0.2)


- Is there any population for which the estimated admixture proportions do not seem to have a good fit?
- Looking at the admixture proportions plot, can you think of a reason why that population might not be correctly modelled?

### Quiz

In [ ]:
from jupyterquiz import display_quiz

display_quiz("https://raw.githubusercontent.com/popgenDK/courses/main/current_exercises/admixture/quiz/admix_quiz4.json")


## Trying other values of K

We will now do again the analyses but using 4 instead of 3 ancestral populations. We start by doing 20 independent runs of NGSadmix, with the same settings except that this time we use K = 4. 

We then collect the likelihoods form the log files and look at them to assess if the optimization has converged to the global maximum likelihood.

Again, the analyses has already been run and we just provide the code to print the commands: 



In [ ]:
source env.sh   # paths + cd to the working folder

inputpath=1000G5pops.inputgl.beagle.gz
outpath=admixoutput
K=4

for i in `seq 1 20`
do
    echo "$NGSadmix -likes $inputpath -K $K -P 10 -minMaf 0.05 -seed $i -o ${outpath}/1000G5popsAdmixK${K}seed${i}"
done

outadmix=admixoutput/1000G5popsAdmix
rm -f allK${K}.likes
for i in `seq 1 20`
do 
    cat ${outadmix}K${K}seed$i.log | grep "best like" | awk -F"[ =]" '{print $3}' >> allK${K}.likes
done


- Can you spot the how the code is different from the code for K=3?

Now try to looks at the log likelihoods for the 20 runs sorted from best (top) to worst (bottom):

In [ ]:
source env.sh   # paths + cd to the working folder

cat -n allK${K}.likes | sort -rhk2


- Does it look we reached convergence for K=4?

We will now run evalAdmix to assess the model fit of the best admixture run (again, it has been pre run and we just print the command):

In [ ]:
source env.sh   # paths + cd to the working folder

K=4
besti=9
inbgl=1000G5pops.inputgl.beagle.gz
inadmix=admixoutput/1000G5popsAdmixK${K}seed${besti}
out=evaladmixoutput/1000G5pops.K${K}seed${besti}.corres

echo "$EVALADMIX -beagle $inbgl -fname ${inadmix}.fopt.gz -qname ${inadmix}.qopt -o $out -P 20"


**Questions**
 - Does a higher K always give a better fit? Can you use the likelihood to choose K?
 - Two seeds at the same K give different likelihoods. Which run do you keep, and why?

We will now visualize the estimated admixture proporitons and the correlation of residuals to assess their fit:

In [ ]:

source("https://raw.githubusercontent.com/GenisGE/evalAdmix/master/visFuns.R") # import some funcitons to help in visualization
pop<-read.table("1000G5pops.pop.info",as.is=T)
q<-read.table("admixoutput/1000G5popsAdmixK4seed9.qopt")


ord<-orderInds(pop = pop[,1], q=q) # sort indiivduals by population and within populaoitn by admixture proportion

# plot admixture proportions
barplot(t(q)[,ord],col=c(5,4,2,3),space=0,border=NA,xlab="Individuals",ylab="Admixture proportions")
text(sort(tapply(1:nrow(pop),pop[ord,1],mean)),-0.05,unique(pop[ord,1]),xpd=T) # add population labels
abline(v=cumsum(sapply(unique(pop[ord,1]),function(x){sum(pop[ord,1]==x)})),col=1,lwd=1.2)

r <- as.matrix(read.table("evaladmixoutput/1000G5pops.K4seed9.corres"))
plotCorRes(r, pop=pop[,1], ord=ord, max_z = 0.2)


- What population does the new cluster that we have added correspond to?
- Based on the correlation of residuals, would you say adding that cluster has given a significant improvement to the model fit?


# B. Use of fastNGSadmix to infer admixture proportions for 3 samples 
Let's try to use fastNGSadmix. Specifically, let's see if we can use it to infer the ancestry of 3 samples: sample1, sample2 and sample3.

## Setup paths

To this first setup some paths by typing this in the terminal:

### How fastNGSadmix differs

NGSadmix estimated the ancestral allele frequencies and the admixture proportions at the same time,
which is only possible because we had hundreds of samples. With a single sample that is hopeless -
there is no way to tell an unusual individual from an unusual population.

fastNGSadmix therefore takes the ancestral allele frequencies as *given*, from a reference panel of
populations that someone has already characterised, and estimates only the admixture proportions of
the one sample. That makes it fast enough to run on a laptop, and it is how ancestry is inferred for
single ancient or forensic samples. The price is that the answer can only ever be phrased in terms of
the populations that are in the panel, which is something we will test directly below.